In [1]:
import numpy as np
import polars as pl
import plotly 
import scipy 
import typing
from scipy import integrate, stats

In [24]:
conf_game = pl.read_csv("MConferenceTourneyGames.csv")
print(conf_game)

shape: (6_793, 5)
┌────────┬────────────┬────────┬─────────┬─────────┐
│ Season ┆ ConfAbbrev ┆ DayNum ┆ WTeamID ┆ LTeamID │
│ ---    ┆ ---        ┆ ---    ┆ ---     ┆ ---     │
│ i64    ┆ str        ┆ i64    ┆ i64     ┆ i64     │
╞════════╪════════════╪════════╪═════════╪═════════╡
│ 2001   ┆ ovc        ┆ 120    ┆ 1122    ┆ 1369    │
│ 2001   ┆ ovc        ┆ 120    ┆ 1183    ┆ 1287    │
│ 2001   ┆ ovc        ┆ 120    ┆ 1293    ┆ 1398    │
│ 2001   ┆ ovc        ┆ 120    ┆ 1399    ┆ 1404    │
│ 2001   ┆ a_sun      ┆ 121    ┆ 1194    ┆ 1144    │
│ …      ┆ …          ┆ …      ┆ …       ┆ …       │
│ 2025   ┆ sec        ┆ 132    ┆ 1196    ┆ 1397    │
│ 2025   ┆ aac        ┆ 132    ┆ 1272    ┆ 1412    │
│ 2025   ┆ big_ten    ┆ 132    ┆ 1276    ┆ 1458    │
│ 2025   ┆ a_ten      ┆ 132    ┆ 1433    ┆ 1206    │
│ 2025   ┆ ivy        ┆ 132    ┆ 1463    ┆ 1165    │
└────────┴────────────┴────────┴─────────┴─────────┘


In [25]:
regular_season_df = pl.read_csv("MRegularSeasonCompactResults.csv")
regular_season_df = regular_season_df.with_columns(pl.lit("regular_season").alias("which"))
march_madness = pl.read_csv("MNCAATourneyCompactResults.csv")
march_madness = march_madness.with_columns(pl.lit("march_madness").alias("which"))

team_mapping = pl.read_csv("MTeams.csv")

all_records_df = pl.concat((regular_season_df, march_madness), how="vertical").sort("Season", descending=False)



In [38]:
def update_vanilla_elo(team_a_elo, team_b_elo):
    win_prob = 1 / (1 + 10**((team_b_elo - team_a_elo) / 400))
    k_factor = 100
    change_in_elo = k_factor * (1 - win_prob)
    return change_in_elo

In [39]:
elos: dict = dict()
team_ids = team_mapping["TeamID"].to_list()
for id in team_ids:
    elos[id] = 1500.00

# need to loop through manually due to skip  
for row in all_records_df.iter_rows():

        win_team_id = row[2]
        lose_team_id = row[4]

        win_team_elo = elos[win_team_id]
        lose_team_elo = elos[lose_team_id]
        
        change_in_elo = update_vanilla_elo(win_team_elo, lose_team_elo)

        # update winning team elo
        elos[win_team_id] = elos[win_team_id] + change_in_elo

        # update lose team elo
        elos[lose_team_id] = elos[lose_team_id] - change_in_elo


vanilla_elos_df = pl.DataFrame({"team_id": elos.keys()})
vanilla_elos_df = pl.concat((vanilla_elos_df, pl.DataFrame({"vanilla_elo": elos.values()})), how="horizontal")

In [40]:
print(vanilla_elos_df)

shape: (381, 2)
┌─────────┬─────────────┐
│ team_id ┆ vanilla_elo │
│ ---     ┆ ---         │
│ i64     ┆ f64         │
╞═════════╪═════════════╡
│ 1101    ┆ 1284.14471  │
│ 1102    ┆ 1158.026424 │
│ 1103    ┆ 1844.267534 │
│ 1104    ┆ 2153.83113  │
│ 1105    ┆ 1176.329282 │
│ …       ┆ …           │
│ 1477    ┆ 1122.869882 │
│ 1478    ┆ 1216.436951 │
│ 1479    ┆ 1220.346405 │
│ 1480    ┆ 1145.413925 │
│ 1481    ┆ 1204.095825 │
└─────────┴─────────────┘


In [42]:
def silver_elo(team_a_elo, team_b_elo, actual_margin, K,
               hca_a=0, dist_a=0, dist_b=0, 
               neutral=False, tournament=False):
    
    # 1. K boost for tournament games
    if tournament:
        K = K * 1.25

    # 2. Adjusted Elo difference with home court and travel
    home_bonus = hca_a if not neutral else 0
    travel_a = 8 * dist_a**(1/3) if neutral else 0
    travel_b = 8 * dist_b**(1/3)
    
    delta_elo = team_a_elo + home_bonus - team_b_elo + travel_b - travel_a
    
    # 3. Tournament multiplier on Elo difference
    if tournament:
        delta_elo = delta_elo * 1.07

    # use adjusted delta_elo for prediction
    win_prob = 1 / (1 + 10**(-delta_elo / 400))
    mu = delta_elo / 27
        
    sigma = 11
    
    def integrand(s):
        return (3 + abs(s))**0.85 * stats.norm.pdf(s, mu, sigma)

    N, _ = integrate.quad(integrand, -100, 100)

    S = 1 if actual_margin > 0 else 0

    change_in_elo = K * (3 + abs(actual_margin)) ** 0.85 / N * (S - win_prob)

    # update raw elo, not adjusted — home court and travel are temporary
    return change_in_elo

In [43]:

# Load distance data and build lookup
game_distances = pl.read_csv("game_distances.csv")
dist_lookup: dict[tuple, tuple[float, float]] = {}
for r in game_distances.iter_rows(named=True):
    key = (r["Season"], r["DayNum"], r["WTeamID"], r["LTeamID"])
    dist_lookup[key] = (r["WTeamDist"] or 0.0, r["LTeamDist"] or 0.0)

# Build conference tournament game set for neutral detection
conf_tourney_set: set[tuple] = set()
for r in conf_game.iter_rows(named=True):
    conf_tourney_set.add((r["Season"], r["DayNum"], r["WTeamID"], r["LTeamID"]))

# Build team → conference mapping per season
team_conf_df = pl.read_csv("MTeamConferences.csv")
team_conference: dict[tuple[int, int], str] = {}  # (season, team_id) -> conf
for r in team_conf_df.iter_rows(named=True):
    team_conference[(r["Season"], r["TeamID"])] = r["ConfAbbrev"]

# ── Kalman filter for per-team home court advantage ──
# State: HCA in margin points per team
# Observation: residual = actual_margin - expected_margin (from home team's perspective)
# We track in margin points, convert to ELO via * 27 when passing to silver_elo

HCA_INIT = 3.5          # prior: ~3.5 point home court advantage
HCA_INIT_VAR = 10.0     # initial uncertainty (σ ≈ 3.2 margin points)
HCA_Q_GAME = 0.01       # process noise per game (very slow drift within season)
HCA_Q_SEASON = 4.0      # extra uncertainty at season boundary (roster turnover)
HCA_R = 121.0           # measurement noise (σ = 11 margin points)

hca_mean: dict[int, float] = {}   # team_id -> estimated HCA in margin points
hca_var: dict[int, float] = {}    # team_id -> variance of estimate

def init_hca(team_id: int):
    if team_id not in hca_mean:
        hca_mean[team_id] = HCA_INIT
        hca_var[team_id] = HCA_INIT_VAR

def update_hca(home_team_id: int, actual_margin: float, expected_margin_no_hca: float):
    """Kalman update after a home game. actual_margin from home team's perspective."""
    init_hca(home_team_id)
    # Predict
    pred_mean = hca_mean[home_team_id]
    pred_var = hca_var[home_team_id] + HCA_Q_GAME
    # Innovation: how far off was reality from (expected + our HCA estimate)?
    innovation = actual_margin - expected_margin_no_hca - pred_mean
    # Update
    K = pred_var / (pred_var + HCA_R)
    hca_mean[home_team_id] = pred_mean + K * innovation
    hca_var[home_team_id] = (1 - K) * pred_var

def get_hca_elo(team_id: int) -> float:
    """Get current HCA estimate in ELO points."""
    init_hca(team_id)
    return hca_mean[team_id] * 27  # convert margin points → ELO points

def season_reset_hca():
    """Increase uncertainty at season boundary."""
    for tid in hca_mean:
        hca_var[tid] = hca_var[tid] + HCA_Q_SEASON

# Initialize ELOs and per-season game counters
elos: dict = dict()
team_ids = team_mapping["TeamID"].to_list()
for id in team_ids:
    elos[id] = 1500.00

game_counts: dict[tuple[int, int], int] = {}  # (season, team_id) -> games played

def get_k(season: int, team_id: int) -> float:
    """K ramps linearly from 56 → 38 over games 1-20, then stays at 38."""
    n = game_counts.get((season, team_id), 0)
    if n < 20:
        return 56 - (56 - 38) * (n / 20)  # linear from 56 to 38
    return 38.0

def apply_mean_reversion(elos: dict, season: int):
    """Revert each team 33% toward its conference's mean Elo."""
    conf_teams: dict[str, list[int]] = {}
    for tid in elos:
        conf = team_conference.get((season, tid))
        if conf:
            conf_teams.setdefault(conf, []).append(tid)
    conf_means: dict[str, float] = {}
    for conf, tids in conf_teams.items():
        conf_means[conf] = np.mean([elos[t] for t in tids])
    for tid in elos:
        conf = team_conference.get((season, tid))
        if conf and conf in conf_means:
            elos[tid] = 0.67 * elos[tid] + 0.33 * conf_means[conf]
        else:
            elos[tid] = 0.67 * elos[tid] + 0.33 * 1500

prev_season = None
total_games = len(all_records_df)

# need to loop through manually due to skip  
for i, row in enumerate(all_records_df.iter_rows()):
        season = row[0]
        day_num = row[1]
        win_team_id = row[2]
        win_score = row[3]
        lose_team_id = row[4]
        lose_score = row[5]
        w_loc = row[6]
        which = row[8]

        # Mean reversion + reset game counts + HCA uncertainty bump on new season
        if season != prev_season:
            if prev_season is not None:
                apply_mean_reversion(elos, prev_season)
                season_reset_hca()
                print(f"  Season {prev_season} done")
            game_counts = {k: v for k, v in game_counts.items() if k[0] == season}
            prev_season = season

        win_team_elo = elos[win_team_id]
        lose_team_elo = elos[lose_team_id]
        margin = win_score - lose_score

        # Determine game type
        is_march_madness = which == "march_madness"
        is_conf_tourney = (season, day_num, win_team_id, lose_team_id) in conf_tourney_set

        neutral = is_march_madness or is_conf_tourney or w_loc == "N"
        tournament = is_march_madness

        # K ramps down over first 20 games per team per season
        k_w = get_k(season, win_team_id)
        k_l = get_k(season, lose_team_id)
        K = k_w if k_w == k_l else (k_w + k_l) / 2

        # Get travel distances (data available from 2010+)
        game_key = (season, day_num, win_team_id, lose_team_id)
        if game_key in dist_lookup:
            w_dist, l_dist = dist_lookup[game_key]
        else:
            w_dist, l_dist = 0.0, 0.0

        # silver_elo assumes team_a is the home team when neutral=False
        if w_loc == "A":
            # loser is home (team_a), winner is away (team_b)
            home_id = lose_team_id
            hca_elo = get_hca_elo(home_id)
            change = silver_elo(
                lose_team_elo, win_team_elo, -margin, K,
                hca_a=hca_elo, dist_a=l_dist, dist_b=w_dist,
                neutral=neutral, tournament=tournament,
            )
            elos[lose_team_id] = elos[lose_team_id] + change
            elos[win_team_id] = elos[win_team_id] - change
            # Kalman update: home team is loser, margin from home perspective = -margin
            expected_margin_no_hca = (lose_team_elo - win_team_elo) / 27
            update_hca(home_id, -margin, expected_margin_no_hca)

        elif w_loc == "H":
            # winner is home (team_a)
            home_id = win_team_id
            hca_elo = get_hca_elo(home_id)
            change = silver_elo(
                win_team_elo, lose_team_elo, margin, K,
                hca_a=hca_elo, dist_a=w_dist, dist_b=l_dist,
                neutral=neutral, tournament=tournament,
            )
            elos[win_team_id] = elos[win_team_id] + change
            elos[lose_team_id] = elos[lose_team_id] - change
            # Kalman update: home team is winner, margin from home perspective = +margin
            expected_margin_no_hca = (win_team_elo - lose_team_elo) / 27
            update_hca(home_id, margin, expected_margin_no_hca)

        else:
            # WLoc='N': neutral site, no HCA
            change = silver_elo(
                win_team_elo, lose_team_elo, margin, K,
                hca_a=0, dist_a=w_dist, dist_b=l_dist,
                neutral=neutral, tournament=tournament,
            )
            elos[win_team_id] = elos[win_team_id] + change
            elos[lose_team_id] = elos[lose_team_id] - change

        # increment game counts
        game_counts[(season, win_team_id)] = game_counts.get((season, win_team_id), 0) + 1
        game_counts[(season, lose_team_id)] = game_counts.get((season, lose_team_id), 0) + 1

        if (i + 1) % 10000 == 0:
            print(f"  Processed {i + 1}/{total_games} games...")

print(f"  Done — {total_games} games processed")

silver_elo_df = pl.DataFrame({"team_id": list(elos.keys()), "elo": list(elos.values())})
print(silver_elo_df)

# Show some HCA estimates
hca_df = pl.DataFrame({
    "team_id": list(hca_mean.keys()),
    "hca_margin": list(hca_mean.values()),
    "hca_var": [hca_var[t] for t in hca_mean.keys()],
}).join(team_mapping.select("TeamID", "TeamName"), left_on="team_id", right_on="TeamID", how="left")
print("\nTop 10 HCA:")
print(hca_df.sort("hca_margin", descending=True).head(10).select("TeamName", "hca_margin", "hca_var"))
print("\nBottom 10 HCA:")
print(hca_df.sort("hca_margin", descending=False).head(10).select("TeamName", "hca_margin", "hca_var"))


  Season 1985 done
  Season 1986 done
  Processed 10000/199408 games...
  Season 1987 done
  Season 1988 done
  Season 1989 done
  Processed 20000/199408 games...
  Season 1990 done
  Season 1991 done
  Processed 30000/199408 games...
  Season 1992 done
  Season 1993 done
  Processed 40000/199408 games...
  Season 1994 done
  Season 1995 done
  Season 1996 done
  Processed 50000/199408 games...
  Season 1997 done
  Season 1998 done
  Processed 60000/199408 games...
  Season 1999 done
  Season 2000 done
  Processed 70000/199408 games...
  Season 2001 done
  Season 2002 done
  Season 2003 done
  Processed 80000/199408 games...
  Season 2004 done
  Season 2005 done
  Processed 90000/199408 games...
  Season 2006 done
  Season 2007 done
  Processed 100000/199408 games...
  Season 2008 done
  Season 2009 done
  Processed 110000/199408 games...
  Season 2010 done
  Processed 120000/199408 games...
  Season 2011 done
  Season 2012 done
  Processed 130000/199408 games...
  Season 2013 done
  S

In [44]:
print(silver_elo_df)

shape: (381, 2)
┌─────────┬─────────────┐
│ team_id ┆ elo         │
│ ---     ┆ ---         │
│ i64     ┆ f64         │
╞═════════╪═════════════╡
│ 1101    ┆ 1343.661685 │
│ 1102    ┆ 1281.81172  │
│ 1103    ┆ 1745.573087 │
│ 1104    ┆ 2021.986632 │
│ 1105    ┆ 1116.31803  │
│ …       ┆ …           │
│ 1477    ┆ 1203.160748 │
│ 1478    ┆ 1288.418601 │
│ 1479    ┆ 1304.791159 │
│ 1480    ┆ 1207.679182 │
│ 1481    ┆ 1328.655619 │
└─────────┴─────────────┘


In [45]:
print(silver_elo_df)

shape: (381, 2)
┌─────────┬─────────────┐
│ team_id ┆ elo         │
│ ---     ┆ ---         │
│ i64     ┆ f64         │
╞═════════╪═════════════╡
│ 1101    ┆ 1343.661685 │
│ 1102    ┆ 1281.81172  │
│ 1103    ┆ 1745.573087 │
│ 1104    ┆ 2021.986632 │
│ 1105    ┆ 1116.31803  │
│ …       ┆ …           │
│ 1477    ┆ 1203.160748 │
│ 1478    ┆ 1288.418601 │
│ 1479    ┆ 1304.791159 │
│ 1480    ┆ 1207.679182 │
│ 1481    ┆ 1328.655619 │
└─────────┴─────────────┘


In [46]:
silver_elo_df.write_csv("silver_elo_hca_applied.csv")

In [ ]:

all_games = pl.concat((regular_season, march_madness))

all_games = all_games.filter(pl.col("Season") >= 2003)
all_games = all_games.filter(~((pl.col("Season") == 2003) & (pl.col("DayNum") < 36)))

# need to get elo the day before as the elo shouldnt be updated yet
all_games = all_games.with_columns((pl.col("DayNum") - 1).alias("DayNum"))


all_games = all_games.join(
    silver_elo_daily.select("Season", "TeamID", "SilverELO", "DayNum"),
    left_on=["Season", "DayNum", "WTeamID"],
    right_on=["Season", "DayNum", "TeamID"],
    how="left"
).rename({"SilverELO": "W_Elo"})


all_games = all_games.join(
    silver_elo_daily.select("Season", "TeamID", "SilverELO", "DayNum"),
    left_on=["Season", "DayNum", "LTeamID"],
    right_on=["Season", "DayNum", "TeamID"],
    how="left"
).rename({"SilverELO": "L_Elo"})


all_games_win = all_games.with_columns((pl.col("WScore") - pl.col("LScore")).alias("point margin"))
all_games_lose = all_games.with_columns((pl.col("LScore") - pl.col("WScore")).alias("point margin"))
3
all_games_win = all_games_win.rename({"WTeamID": "Team1", "LTeamID": "Team2", "W_Elo": "Team1Elo", "L_Elo": "Team2Elo"})
all_games_lose = all_games_lose.rename({"WTeamID": "Team2", "LTeamID": "Team1", "W_Elo": "Team2Elo", "L_Elo": "Team1Elo"})

cols = all_games_win.columns
all_games = pl.concat([all_games_win.select(cols), all_games_lose.select(cols)])
wide_rankings = others_prediction.pivot(
    on="SystemName",
    index=["Season", "RankingDayNum", "TeamID"],
    values="OrdinalRank"
)

all_games

In [ ]:
all_games = pl.concat((regular_season, march_madness))

all_games = all_games.filter(pl.col("Season") >= 2003)

# --- Silver ELO via asof join ---
silver_elo_sorted = (
    silver_elo_daily.select("Season", "TeamID", "SilverELO", "DayNum")
    .sort("DayNum")
)

all_games = all_games.sort("DayNum").join_asof(
    silver_elo_sorted.rename({"TeamID": "WTeamID", "SilverELO": "W_Elo"}),
    on="DayNum",
    by=["Season", "WTeamID"],
    strategy="backward"
)

all_games = all_games.sort("DayNum").join_asof(
    silver_elo_sorted.rename({"TeamID": "LTeamID", "SilverELO": "L_Elo"}),
    on="DayNum",
    by=["Season", "LTeamID"],
    strategy="backward"
)

# --- Double the dataset (swap winner/loser) ---
all_games_win = all_games.with_columns(
    (pl.col("WScore") - pl.col("LScore")).alias("point_margin")
)
all_games_lose = all_games.with_columns(
    (pl.col("LScore") - pl.col("WScore")).alias("point_margin")
)

all_games_win = all_games_win.rename({
    "WTeamID": "Team1", "LTeamID": "Team2",
    "W_Elo": "Team1Elo", "L_Elo": "Team2Elo"
})
all_games_lose = all_games_lose.rename({
    "WTeamID": "Team2", "LTeamID": "Team1",
    "W_Elo": "Team2Elo", "L_Elo": "Team1Elo"
})

cols = all_games_win.columns
all_games = pl.concat([all_games_win.select(cols), all_games_lose.select(cols)])

# --- Rankings: pivot, carry forward last season, forward-fill, asof join ---
top_systems = ["POM", "MOR", "WLK"]

wide_rankings = (
    others_prediction
    .filter(pl.col("SystemName").is_in(top_systems))
    .pivot(
        on="SystemName",
        index=["Season", "RankingDayNum", "TeamID"],
        values="OrdinalRank"
    )
    .sort("Season", "TeamID", "RankingDayNum")
)

# Carry last season's final ranking into next season as day 0
rank_cols = wide_rankings.columns
last_per_season = (
    wide_rankings
    .sort("RankingDayNum")
    .group_by(["Season", "TeamID"])
    .last()
    .with_columns(
        (pl.col("Season") + 1).cast(pl.Int64).alias("Season"),
        pl.lit(0, dtype=pl.Int64).alias("RankingDayNum"),
    )
    .select(rank_cols)
)

wide_rankings = (
    pl.concat([wide_rankings, last_per_season])
    .sort("Season", "TeamID", "RankingDayNum")
    .with_columns(pl.col(top_systems).forward_fill().over("Season", "TeamID"))
    .rename({"RankingDayNum": "DayNum"})
)

# Join for Team1
t1_rankings = wide_rankings.rename(
    {"TeamID": "Team1"} | {s: f"T1_{s}" for s in top_systems}
).sort("DayNum")
all_games = all_games.sort("DayNum").join_asof(
    t1_rankings, on="DayNum", by=["Season", "Team1"], strategy="backward"
)

# Join for Team2
t2_rankings = wide_rankings.rename(
    {"TeamID": "Team2"} | {s: f"T2_{s}" for s in top_systems}
).sort("DayNum")
all_games = all_games.sort("DayNum").join_asof(
    t2_rankings, on="DayNum", by=["Season", "Team2"], strategy="backward"
)

# Drop non-feature columns (keep Season for CV splits)
all_games = all_games.drop(["WScore", "LScore", "WLoc", "NumOT"])
all_games = all_games.filter(~((pl.col("Season") == 2003) & (pl.col("DayNum") < 36)))
all_games = all_games.drop(["DayNum"])
all_games

In [ ]:
all_games = pl.concat((regular_season, march_madness))
all_games = all_games.filter(pl.col("Season") >= 2003)

# --- Build real dates from Season + DayNum ---
seasons_df = pl.read_csv("MSeasons.csv").select("Season", "DayZero")
seasons_df = seasons_df.with_columns(
    pl.col("DayZero").str.strptime(pl.Date, "%m/%d/%Y")
)

# Add real date to games: DayZero + DayNum days, then shift -1 for pre-game state
all_games = all_games.join(seasons_df, on="Season", how="left")
all_games = all_games.with_columns(
    (pl.col("DayZero") + pl.duration(days=pl.col("DayNum")) - pl.duration(days=1)).alias("GameDate")
).drop("DayZero")

# --- Add real date to Silver ELO snapshots ---
silver_elo_dated = (
    silver_elo_daily
    .select("Season", "TeamID", "SilverELO", "DayNum")
    .join(seasons_df, on="Season", how="left")
    .with_columns(
        (pl.col("DayZero") + pl.duration(days=pl.col("DayNum"))).alias("GameDate")
    )
    .drop("DayZero", "DayNum", "Season")
    .sort("GameDate")
)

all_games = all_games.sort("GameDate").join_asof(
    silver_elo_dated.rename({"TeamID": "WTeamID", "SilverELO": "W_Elo"}),
    on="GameDate",
    by="WTeamID",
    strategy="backward"
)

all_games = all_games.sort("GameDate").join_asof(
    silver_elo_dated.rename({"TeamID": "LTeamID", "SilverELO": "L_Elo"}),
    on="GameDate",
    by="LTeamID",
    strategy="backward"
)

# --- Double the dataset (swap winner/loser) ---
all_games_win = all_games.with_columns(
    (pl.col("WScore") - pl.col("LScore")).alias("point_margin")
)
all_games_lose = all_games.with_columns(
    (pl.col("LScore") - pl.col("WScore")).alias("point_margin")
)

all_games_win = all_games_win.rename({
    "WTeamID": "Team1", "LTeamID": "Team2",
    "W_Elo": "Team1Elo", "L_Elo": "Team2Elo"
})
all_games_lose = all_games_lose.rename({
    "WTeamID": "Team2", "LTeamID": "Team1",
    "W_Elo": "Team2Elo", "L_Elo": "Team1Elo"
})

cols = all_games_win.columns
all_games = pl.concat([all_games_win.select(cols), all_games_lose.select(cols)])

# --- Rankings: pivot, add real dates, forward-fill, asof join ---
top_systems = ["POM", "MOR", "WLK"]

wide_rankings = (
    others_prediction
    .filter(pl.col("SystemName").is_in(top_systems))
    .pivot(
        on="SystemName",
        index=["Season", "RankingDayNum", "TeamID"],
        values="OrdinalRank"
    )
    .join(seasons_df, on="Season", how="left")
    .with_columns(
        (pl.col("DayZero") + pl.duration(days=pl.col("RankingDayNum"))).alias("GameDate")
    )
    .drop("DayZero", "RankingDayNum", "Season")
    .sort("TeamID", "GameDate")
    .with_columns(pl.col(top_systems).forward_fill().over("TeamID"))
    .sort("GameDate")
)

# Join for Team1
t1_rankings = wide_rankings.rename(
    {"TeamID": "Team1"} | {s: f"T1_{s}" for s in top_systems}
)
all_games = all_games.sort("GameDate").join_asof(
    t1_rankings.sort("GameDate"), on="GameDate", by="Team1", strategy="backward"
)

# Join for Team2
t2_rankings = wide_rankings.rename(
    {"TeamID": "Team2"} | {s: f"T2_{s}" for s in top_systems}
)
all_games = all_games.sort("GameDate").join_asof(
    t2_rankings.sort("GameDate"), on="GameDate", by="Team2", strategy="backward"
)

# Drop non-feature columns (keep Season for CV splits)
all_games = all_games.drop(["GameDate", "WScore", "LScore", "WLoc", "NumOT"])
all_games = all_games.filter(~((pl.col("Season") == 2003) & (pl.col("DayNum") <= 37)))
all_games = all_games.drop(["DayNum"])
all_games